# Baseline Modelling — Polymarket → Gold 5-min Returns

Pipeline summary (this notebook covers steps 1-9 of the design brief):

1. **Load** the latest `gold_panel_<date>.csv` produced by the feature-engineering notebook.
2. **Load** the Bloomberg target (`GOLD USD SPOT PER OZ`, `Close`).
3. Build the target `y`: **log-return over the past 5 min, lagged −1** so that predictors at `t` predict the return realised between `t` and `t+1` (≈ +5 min).
4. Attach **auto-regressive (AR) features** to the predictors.
5. Build `X`, `y` (scikit-learn ready). Build `X_traditional` from the remaining Bloomberg indicators (loaded but not used for fitting yet).
6. **Flexible walk-forward modelling framework** with three models (Linear, LSTM, Random Forest) × four window schemes (fixed 120 / 240 / 300 + expanding). LSTM uses **batched test blocks (12 obs per retrain)** to stay tractable.
7. **Walk-forward evaluation** (train on window → predict next observation, or next 12 for LSTM).
8. **Logging**: every run appends a row to `Results/runs_log.txt` plus a detailed `.json` side-car (model spec, metrics, data hash, run timestamp).
9. **Persistence**: models are saved as `Models/<MODEL>_<WINDOW>_<DATA-DATE>.pkl|.keras`. If a matching artefact already exists and the input CSV hasn't changed, training is skipped and the model is reloaded.

> **Open questions / critical issues are collected in `NOTES_modelling_baseline.md` — please read it before interpreting results.**


## 0. Config

In [ ]:
# %% ── CELL 0 : CONFIG ───────────────────────────────────────────────────────
from pathlib import Path

# Folders
DATA_DIR     = Path("./Data")
MODELS_DIR   = Path("./Models")
RESULTS_DIR  = Path("./Results")
for p in (MODELS_DIR, RESULTS_DIR):
    p.mkdir(exist_ok=True, parents=True)

# File patterns
GOLD_PANEL_GLOB = "gold_panel_*.csv"          # dynamic date in file name
BLOOMBERG_XLSX_CANDIDATES = [
    DATA_DIR / "Indicators Data bloomberg.xlsx",
    DATA_DIR / "Indicators-Data-bloomberg.xlsx",
    DATA_DIR / "Indicators_Data_bloomberg.xlsx",
]
TARGET_SHEET    = "GOLD USD SPOT PER OZ"
TARGET_COL      = "Close"

def resolve_bloomberg_xlsx(candidates: list[Path], data_dir: Path) -> Path:
    for p in candidates:
        if p.exists():
            return p
    # Fallback: pick any workbook that looks like the indicators file
    wildcard = sorted(data_dir.glob("*Indicators*Data*bloomberg*.xlsx"))
    if wildcard:
        return wildcard[0]
    tried = "\n  - ".join(str(p) for p in candidates)
    raise FileNotFoundError(
        "Bloomberg workbook not found. Tried:\n"
        f"  - {tried}\n"
        "Expected something like 'Indicators Data bloomberg.xlsx' in ./Data."
    )

BLOOMBERG_XLSX = resolve_bloomberg_xlsx(BLOOMBERG_XLSX_CANDIDATES, DATA_DIR)
print(f"Using Bloomberg workbook: {BLOOMBERG_XLSX}")

# ─── Modelling ───────────────────────────────────────────────────────────────
BAR_MINUTES         = 5          # native cadence of the Bloomberg close series
RETURN_HORIZON_MIN  = 15         # ⟵ target-return horizon in MINUTES.
                                 #    60   -> 1-hour returns (current default)
                                 #    240  -> 4-hour returns (to test next)
                                 #    1440 -> next-day returns (to test later)
                                 #    5    -> old 5-min behaviour
HORIZON_STEPS       = max(1, RETURN_HORIZON_MIN // BAR_MINUTES)  # #bars ahead predicted
AR_LAGS             = [1, 2, 3, 6, 12]       # past-return lags (in 5-min bars)
AR_MA_WINDOWS       = [3, 6, 12, 36]         # rolling-mean windows on past returns
WINDOW_SCHEMES  = {                          # {label: ("fixed"|"expanding", size)}
    "fixed120":  ("fixed", 120),
    "fixed240":  ("fixed", 240),
    "fixed300":  ("fixed", 300),
    "expanding": ("expanding", 120),         # 120 = minimum training size before first prediction
}
LSTM_TEST_BLOCK = 12           # LSTM retrain cadence — predict next 12 obs per refit
LSTM_SEQ_LEN    = 12           # look-back length fed into the LSTM (≈ 1 hour)
LSTM_EPOCHS     = 8
LSTM_BATCH      = 32
RF_N_ESTIMATORS = 200
RANDOM_STATE    = 67

# ─── Regularisation knobs (tunable from here for quick iteration) ────────────
# Ridge penalty. Start around 10–100 when Polymarket features are still raw-ish;
# drop to ~1.0 once proper feature engineering / selection has been done.
RIDGE_ALPHA     = 10.0

# PCA pre-compression applied to the Polymarket block ONLY (AR features are
# kept raw, they are already in return-space and low-dim).
# Set PCA_N_COMPONENTS = 0 (or None) to disable PCA entirely.
#   typical range: 20–40 while Polymarket dim is high;
#   after feature engineering you can probably drop it to 0.
PCA_N_COMPONENTS = None

# ─── Traditional indicators — separate PCA and feature engineering controls ──
# Independent PCA for the Bloomberg traditional block: typically 3–8 components
# (one per macro factor: rates, equities, FX, commodities, vol, etc.).
# Set to 0 to disable PCA on the traditional block entirely.
PCA_N_COMPONENTS_TRADITIONAL = None

# Forward-fill cap when resampling daily Bloomberg indicators to the 5-min gold
# clock.  None = carry indefinitely (standard for Bloomberg daily data);
# set an integer to cap stale carries (e.g., 288 = ~1 trading day on a 5-min grid).
TRADITIONAL_MAX_FFILL_BARS   = None

# Add a staleness counter feature for each traditional indicator:
# tells the model how many 5-min bars have elapsed since the last real tick.
# Useful because daily series are stale for ~288 bars between updates.
TRADITIONAL_USE_STALENESS    = True

# Drop raw price-level columns after stationarising (default True).
# They are collinear with the diff/log-return columns and can confuse Ridge.
TRADITIONAL_DROP_PRICE_LEVELS = True

# AR features of the gold log-return appended to the traditional predictor matrix.
# Must mirror the full AR feature set used for the polymarket block so that both
# sets of models see the same autoregressive information:
#   - 5 lagged returns  (AR_LAGS)
#   - 4 rolling means   (AR_MA_WINDOWS)
#   - 2 rolling stds    ([12, 36])
# These are passed separately to build_X_traditional via ar_lags / ar_ma_windows /
# ar_vol_windows; keeping them as explicit config allows independent tuning later.
TRADITIONAL_AR_LAGS       = [1, 2, 3, 6, 12]   # must match AR_LAGS
# ar_ma_windows and ar_vol_windows for the trad block are passed as AR_MA_WINDOWS
# and [12, 36] directly in Cell 5 (they share the same config as the poly block).

# ─── Feature engineering checkmark (variance prefilter gate) ─────────────────
# NOTE: the variance pre-filter in Cell 4.1 is a stop-gap that keeps LSTM/Ridge
# tractable while Polymarket features are still raw and high-dimensional. Once
# proper feature engineering AND selection have been done upstream (on the
# polymarket panel), the prefilter is no longer needed and can be skipped:
#   - FEATURE_ENGINEERING_DONE = False  -> prefilter ON  (keep top-N by variance)
#   - FEATURE_ENGINEERING_DONE = True   -> prefilter OFF (use all columns)
FEATURE_ENGINEERING_DONE = False

MAX_FEATURES_PREFILTER = 150   # crude variance prefilter (only used if FE not done)

# ─── Daily gap handling (Bloomberg ~23:00 break, see Cell 10) ────────────────
# Adds an `is_post_break` dummy AND a `poly_movement_during_break` feature
# (approach 2 + 3 combined — see the PDF diagnosis). Turn off to revert.
HANDLE_DAILY_GAP = True
GAP_THRESHOLD    = "60min"       # a bar-to-bar step > this counts as a session break

# Evaluation toggles
FORCE_RETRAIN   = False        # set True to ignore cached models
DRY_RUN_ROWS    = None         # e.g. 1000 to debug on a slice; None = full


Using Bloomberg workbook: Data\Indicators Data bloomberg.xlsx


## 1. Dependencies

In [2]:
# %% ── CELL 1 : DEPENDENCIES ─────────────────────────────────────────────────
import os, re, json, hashlib, joblib, warnings, datetime as dt, time
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

from sklearn.linear_model import Ridge
from sklearn.ensemble    import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics     import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import VarianceThreshold

# TensorFlow / Keras (LSTM) — imported lazily so the notebook still runs if TF isn't installed
_TF_AVAILABLE = True
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, load_model
    from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
    from tensorflow.keras.callbacks import EarlyStopping
    tf.random.set_seed(RANDOM_STATE)
except Exception as _tf_err:
    _TF_AVAILABLE = False
    print("⚠️  TensorFlow not importable — LSTM runs will be skipped.", _tf_err)

np.random.seed(RANDOM_STATE)
print("✅ Dependencies loaded. TF:", _TF_AVAILABLE)


✅ Dependencies loaded. TF: True


## 1.1 Traditional predictor helpers

Functions for building the Bloomberg traditional predictor matrix:
- **`make_stationary_features`** — log-diff for positive price series, simple diff for others.
- **`build_X_traditional`** — reindexes each Bloomberg sheet to the gold 5-min clock, forward-fills, stationarises, optionally adds staleness counters, then aligns to the modelling index.
- **`fit_transform_pca_fold`** — fold-safe PCA (fit on train, apply to test) used inside the walk-forward loop for the traditional block.

In [3]:
# %% ── CELL 1.1 : TRADITIONAL PREDICTOR HELPERS ─────────────────────────────

def make_stationary_features(df: pd.DataFrame, exclude_cols=None) -> pd.DataFrame:
    """Convert each column to a stationary series:
    - strictly positive series  → log-diff  (captures percentage moves)
    - other numeric series      → simple diff (rates, spreads, indices that cross 0)
    Columns in `exclude_cols` (e.g. staleness counters) are passed through as-is.
    """
    exclude_cols = set(exclude_cols or [])
    out = pd.DataFrame(index=df.index)
    for col in df.columns:
        s = pd.to_numeric(df[col], errors="coerce")
        if col in exclude_cols:
            out[col] = s
            continue
        strictly_positive = (s.dropna() > 0).all()
        if strictly_positive:
            out[f"{col}_logret"] = np.log(s).diff()
        else:
            out[f"{col}_diff"] = s.diff()
    return out


def build_X_traditional(
    bb_dict: dict,
    target_sheet: str,
    master_index: pd.DatetimeIndex,
    align_index: pd.DatetimeIndex,
    max_ffill_bars=None,
    use_staleness: bool = True,
    ar_lags=(1, 2, 3, 6, 12),
    ar_ma_windows=(3, 6, 12, 36),
    ar_vol_windows=(12, 36),
    logret_bar: pd.Series = None,
) -> pd.DataFrame:
    """Build a stationary traditional predictor matrix aligned to `align_index`.

    Pipeline per non-target Bloomberg sheet:
    1. Reindex the close column to `master_index` (5-min gold clock).
    2. Optionally compute a staleness counter (bars since last real tick).
    3. Forward-fill up to `max_ffill_bars`.
    4. Stationarise: log-diff (positive series) or simple diff (other).
    5. Append AR lags, rolling means, and rolling stds of the gold log-return
       (same full AR feature set as build_ar_features for the polymarket block).
    6. Reindex everything to `align_index` (= X.index, valid polymarket rows).

    Reuses the already-loaded `bb` dict — no extra workbook I/O.
    """
    import re as _re
    price_cols_raw = {}   # {feature_name: forward-filled Series}
    stale_cols_raw = {}   # {stale_col_name: counter Series}

    for sheet, df in bb_dict.items():
        if sheet == target_sheet:
            continue
        if "close" not in df.columns:
            continue

        feature_name = _re.sub(r"[^0-9a-zA-Z]+", "_", sheet).strip("_").lower()
        s_raw = df["close"].sort_index()
        s_raw = s_raw[~s_raw.index.duplicated(keep="last")]

        # Align to 5-min master index
        s_aligned = s_raw.reindex(master_index)

        if use_staleness:
            # Count bars since the last real (non-NaN) tick
            is_new_tick       = s_aligned.notna()
            real_tick_groups  = is_new_tick.cumsum()
            stale_count       = (~is_new_tick).groupby(real_tick_groups).cumsum().astype(int)
            stale_cols_raw[f"{feature_name}_stale"] = stale_count

        s_filled = s_aligned.ffill(limit=max_ffill_bars)
        price_cols_raw[feature_name] = s_filled

    if not price_cols_raw:
        return pd.DataFrame(index=align_index)

    price_df   = pd.DataFrame(price_cols_raw,  index=master_index)
    stale_df   = pd.DataFrame(stale_cols_raw,  index=master_index)

    # Stationarise price-level columns
    stationary = make_stationary_features(price_df)

    # Merge staleness counters back (they stay in levels — already stationary-ish)
    if use_staleness and not stale_df.empty:
        stationary = stationary.join(stale_df, how="left")

    # Add the full AR feature set for the gold log-return — mirrors build_ar_features
    # used for the polymarket block (5 lagged returns, 4 rolling means, 2 rolling stds).
    if logret_bar is not None:
        logret_aligned = logret_bar.reindex(master_index)
        for lag in ar_lags:
            stationary[f"trad_gold_lag{lag}"] = logret_aligned.shift(lag)
        for w in ar_ma_windows:
            stationary[f"trad_gold_ma{w}"] = logret_aligned.shift(1).rolling(w).mean()
        for w in ar_vol_windows:
            stationary[f"trad_gold_std{w}"] = logret_aligned.shift(1).rolling(w).std()

    # Final alignment to the modelling index
    stationary = stationary.reindex(align_index)
    stationary = stationary.replace([np.inf, -np.inf], np.nan)
    stationary = stationary.ffill().fillna(0.0)   # 0.0 for series-start NaN

    return stationary


def fit_transform_pca_fold(
    X_train: np.ndarray,
    X_test:  np.ndarray,
    n_components: int,
    prefix: str,
    random_state: int = RANDOM_STATE,
):
    """Fold-safe PCA: fit only on `X_train`, transform both splits.
    Returns raw numpy arrays (not DataFrames) so they drop straight into the
    existing ScaledRegressor / LSTM flow.
    Returns (X_train_out, X_test_out, pca_obj); pca_obj is None if PCA is skipped.
    """
    if (not n_components) or (n_components <= 0) or (X_train.shape[1] <= n_components):
        return X_train, X_test, None
    n_comp = min(n_components, X_train.shape[1], X_train.shape[0])
    pca = PCA(n_components=n_comp, random_state=random_state)
    X_train_out = pca.fit_transform(X_train)
    X_test_out  = pca.transform(X_test)
    return X_train_out, X_test_out, pca

print("✅ Traditional predictor helpers loaded.")


✅ Traditional predictor helpers loaded.


## 2. Locate the latest gold panel CSV

The feature-engineering pipeline writes `gold_panel_<YYYY-MM-DD>.csv` to `./Data`. Here we pick the file with the most recent date in its name (NOT file mtime — mtime can be misleading if the file was merely copied).

In [4]:
# %% ── CELL 2 : LOAD LATEST GOLD PANEL ───────────────────────────────────────
_date_rx = re.compile(r"gold_panel_(\d{4}-\d{2}-\d{2})\.csv$")

def find_latest_panel(folder: Path) -> tuple[Path, str]:
    candidates = []
    for p in folder.glob(GOLD_PANEL_GLOB):
        m = _date_rx.search(p.name)
        if m:
            candidates.append((m.group(1), p))
    if not candidates:
        raise FileNotFoundError(f"No files matching {GOLD_PANEL_GLOB} in {folder}")
    candidates.sort(key=lambda t: t[0])   # lexicographic sort works for ISO dates
    return candidates[-1][1], candidates[-1][0]

panel_path, panel_date = find_latest_panel(DATA_DIR)
print(f"Latest panel: {panel_path.name}  (data date = {panel_date})")

X_raw = pd.read_csv(panel_path, parse_dates=["scraped_at"]).set_index("scraped_at").sort_index()
if DRY_RUN_ROWS:
    X_raw = X_raw.iloc[-DRY_RUN_ROWS:]
print(f"X_raw shape : {X_raw.shape}   range: {X_raw.index.min()} → {X_raw.index.max()}")


Latest panel: gold_panel_2026-04-23.csv  (data date = 2026-04-23)
X_raw shape : (6751, 2329)   range: 2026-03-30 15:15:52.586000 → 2026-04-23 22:00:19.571000


## 3. Load Bloomberg indicators — build target and X_traditional

The Bloomberg workbook has one sheet per indicator, each with 5 columns (`Date, Open, High, Low, Close`). We keep the **Close** column of every sheet. `GOLD USD SPOT PER OZ` becomes the target; the rest go into `X_traditional`.

In [5]:
# %% ── CELL 3.0 : ENSURE OPENPYXL ───────────────────────────────────────────
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("openpyxl") is None:
    print("openpyxl not found. Installing via pip...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
else:
    print("openpyxl is already installed.")


openpyxl is already installed.


In [6]:
# %% ── CELL 3 : LOAD BLOOMBERG SHEETS ────────────────────────────────────────
def load_bloomberg(xlsx_path: Path) -> dict[str, pd.DataFrame]:
    '''Load every Bloomberg sheet that has a `Date` column.
       Some sheets use `Close`, some use `Last Price` — we normalise both to `close`.'''
    xl = pd.ExcelFile(xlsx_path)
    out = {}
    for sheet in xl.sheet_names:
        df = pd.read_excel(xlsx_path, sheet_name=sheet, header=0)
        if df.empty or "Date" not in df.columns:
            continue
        df = df.rename(columns={c: c.strip() for c in df.columns})
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"]).set_index("Date").sort_index()
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        # normalise the price column name
        if "Close" in df.columns:
            df = df.rename(columns={"Close": "close"})
        elif "Last Price" in df.columns:
            df = df.rename(columns={"Last Price": "close"})
        else:
            numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
            if not numeric_cols:
                continue
            df = df.rename(columns={numeric_cols[-1]: "close"})
        out[sheet] = df
    return out

bb = load_bloomberg(BLOOMBERG_XLSX)
print("Bloomberg sheets loaded:", list(bb.keys()))
gold = bb[TARGET_SHEET][["close"]].rename(columns={"close": "gold_close"})
print("Gold close range:", gold.index.min(), "→", gold.index.max(), "| rows:", len(gold))


Bloomberg sheets loaded: ['GOLD USD SPOT PER OZ', 'CRUDE OIL (WTI) FUTURES PRICE', 'CRUDE OIL (BRENT) FUTURES PRICE', 'S&P 500 Index', 'USD Index', 'VIX Index', 'USGG10YR Index']
Gold close range: 2026-03-24 23:00:00 → 2026-04-17 22:55:00 | rows: 4693


### 3.1 Build the target `y`

We want a **regression** target on **future returns**, not prices.

\[
y_t = \log\!\left(\frac{P_{t+1}}{P_t}\right)
\]

so that predictors available at time `t` forecast the realised 5-min log-return over `[t, t+1]`. Equivalently: take the 5-min log-return series and shift it by `-1`.

**Why log-returns vs simple returns?** Additivity across time (log-returns sum), symmetry, and far more stable numerics for small moves — standard in high-frequency finance.

In [7]:
# %% ── CELL 3.1 : BUILD y (log-returns over RETURN_HORIZON_MIN, lagged -HORIZON_STEPS) ───
# ─── FIX (from Operation Fixing Terrible Performance, point 5) ───────────────
# We now use a CONFIGURABLE horizon. The per-bar log-return is still computed
# at the native 5-min cadence (it's what AR features need), but the TARGET is
# the log-return over HORIZON_STEPS bars ahead:
#     y_t = log(P_{t+H}) - log(P_t)
# where H = HORIZON_STEPS = RETURN_HORIZON_MIN / BAR_MINUTES.
#
# Switching RETURN_HORIZON_MIN in the config cell is the only change needed
# to move from 1h -> 4h -> next-day returns.
# Bloomberg sheets list rows newest-first; sort_index() already fixed that.
gold["logret_bar"]     = np.log(gold["gold_close"]).diff()                    # per-bar (5-min) log-return
gold["logret_horizon"] = (np.log(gold["gold_close"])
                          - np.log(gold["gold_close"]).shift(HORIZON_STEPS))  # return realised over [t-H, t]
gold["y"]              = gold["logret_horizon"].shift(-HORIZON_STEPS)         # shift forward = realised over [t, t+H]
# Backwards-compat alias (some downstream cells still reference `logret_5m`).
gold["logret_5m"]      = gold["logret_bar"]
# (gold.index = t. gold["y"].loc[t] = log(P_{t+H}) - log(P_t), i.e. the H-bar-ahead return.)
print(f"Target horizon: {RETURN_HORIZON_MIN} min  ({HORIZON_STEPS} bars of {BAR_MINUTES} min each)")
print(gold[["gold_close","logret_bar","logret_horizon","y"]].tail(6))


Target horizon: 5 min  (1 bars of 5 min each)
                     gold_close  logret_bar  logret_horizon         y
Date                                                                 
2026-04-17 22:30:00     4846.59    0.000140        0.000140 -0.000873
2026-04-17 22:35:00     4842.36   -0.000873       -0.000873 -0.001451
2026-04-17 22:40:00     4835.34   -0.001451       -0.001451 -0.000360
2026-04-17 22:45:00     4833.60   -0.000360       -0.000360  0.000213
2026-04-17 22:50:00     4834.63    0.000213        0.000213 -0.000888
2026-04-17 22:55:00     4830.34   -0.000888       -0.000888       NaN


### 3.2 Auto-regressive (AR) features

**Design choice — rationale:**
Gold returns at 5-min cadence exhibit small but exploitable **short-term momentum / mean-reversion** and strong **volatility clustering**. Any non-trivial baseline must give the model access to its own past, otherwise we're asking Polymarket features alone to beat a signal that's already in the price path.

I'm adding three groups of AR features, all computed on `logret_5m` (which is *already observed* at time `t`, so no look-ahead):

1. **Lagged returns** at 1, 2, 3, 6, 12 bars (5, 10, 15, 30, 60 min). Captures raw momentum.
2. **Rolling means of past returns** over 3, 6, 12, 36 bars. Same idea as the lectures' moving-average baseline; a denoised direction estimate.
3. **Rolling std (realised vol proxy)** over 12 and 36 bars. Lets tree models condition on the current vol regime — classic HAR-RV intuition.

All features use only information up to and including `t`, so there is **no leakage**. They are attached to the `X` predictor matrix after timestamp alignment.

In [8]:
# %% ── CELL 3.2 : AR FEATURES ────────────────────────────────────────────────
# AR features are always built on the NATIVE 5-min log-returns (logret_bar).
# They capture short-term momentum / vol-clustering at the bar cadence; this
# is independent of the target horizon.
def build_ar_features(logret: pd.Series,
                      lags=AR_LAGS,
                      ma_windows=AR_MA_WINDOWS,
                      vol_windows=(12, 36)) -> pd.DataFrame:
    feats = {}
    for L in lags:
        feats[f"ar_ret_lag{L}"] = logret.shift(L)
    for W in ma_windows:
        feats[f"ar_ret_ma{W}"]  = logret.shift(1).rolling(W).mean()
    for W in vol_windows:
        feats[f"ar_ret_std{W}"] = logret.shift(1).rolling(W).std()
    return pd.DataFrame(feats)

ar_feats = build_ar_features(gold["logret_bar"])
print("AR feature matrix shape:", ar_feats.shape)
print(ar_feats.tail(3))


AR feature matrix shape: (4693, 11)
                     ar_ret_lag1  ar_ret_lag2  ar_ret_lag3  ar_ret_lag6  \
Date                                                                      
2026-04-17 22:45:00    -0.001451    -0.000873     0.000140     0.001169   
2026-04-17 22:50:00    -0.000360    -0.001451    -0.000873    -0.000641   
2026-04-17 22:55:00     0.000213    -0.000360    -0.001451    -0.000569   

                     ar_ret_lag12  ar_ret_ma3  ar_ret_ma6  ar_ret_ma12  \
Date                                                                     
2026-04-17 22:45:00     -0.000214   -0.000728   -0.000371    -0.000420   
2026-04-17 22:50:00     -0.000813   -0.000895   -0.000626    -0.000432   
2026-04-17 22:55:00     -0.000025   -0.000533   -0.000483    -0.000347   

                     ar_ret_ma36  ar_ret_std12  ar_ret_std36  
Date                                                          
2026-04-17 22:45:00    -0.000113      0.000658      0.000598  
2026-04-17 22:50:00    -0.00

## 4. Align Polymarket features to Bloomberg's 5-min grid, assemble `X` and `y`

Polymarket is scraped every ~5 min but the timestamps are **not on the clock** (14:15:52, 14:21:07, …), whereas Bloomberg prices sit exactly on `:00/:05/:10/…`. To merge them I use `merge_asof` with a 5-min tolerance and `direction="backward"`: for every Bloomberg bar, I grab the **latest** Polymarket snapshot strictly at or before that bar. This is the only look-ahead-safe way to align the two clocks.

After alignment we drop rows with no target (the last bar and any pre-warmup AR rows).

In [ ]:
# %% ── CELL 4 : ALIGN + BUILD X, y ───────────────────────────────────────────
# 1) Restrict Bloomberg to the period covered by the panel
start, end = X_raw.index.min().floor("5min"), X_raw.index.max().ceil("5min")
gold_aligned = gold.loc[(gold.index >= start) & (gold.index <= end)].copy()
print(f"Bloomberg bars in overlap window: {len(gold_aligned)}")

# 2) merge_asof: for each 5-min Bloomberg bar, pull the latest polymarket snapshot ≤ bar
X_raw_sorted = X_raw.sort_index()
merged = pd.merge_asof(
    left      = gold_aligned.reset_index().rename(columns={"Date":"ts"}),
    right     = X_raw_sorted.reset_index().rename(columns={"scraped_at":"ts"}),
    on        = "ts",
    direction = "backward",
    tolerance = pd.Timedelta(f"{BAR_MINUTES}min"),
).set_index("ts")

# 3) Pull in AR features (index already Bloomberg bars)
merged = merged.join(ar_feats, how="left")

# 4) Separate into X and y
y = merged["y"]
polymarket_cols = [c for c in X_raw.columns]
ar_cols         = list(ar_feats.columns)
feature_cols    = polymarket_cols + ar_cols
X = merged[feature_cols].copy()

# 5) Drop rows where y or AR features are undefined
valid = y.notna() & merged[ar_cols].notna().all(axis=1)
X, y = X.loc[valid], y.loc[valid]

# 6) Column-level NA handling: forward-fill within polymarket cols (stale quote is the truth),
#    then fill remaining with 0 (market that didn't exist yet).
X[polymarket_cols] = X[polymarket_cols].ffill().fillna(0.0)

# NOTE: Stationarity transformation (differencing / log-differencing) is now applied
# upstream in the Feature Engineering notebook before the gold_panel CSV is written.
# The X_raw columns loaded here are therefore already stationary — no further
# differencing is needed at this stage.

print(f"Final X shape: {X.shape}")
print(f"Final y shape: {y.shape}")
print(f"y sample stats: mean={y.mean():.2e}  std={y.std():.2e}  min={y.min():.2e}  max={y.max():.2e}")


Bloomberg bars in overlap window: 3682
Final X shape: (3681, 2340)
Final y shape: (3681,)
y sample stats: mean=1.52e-05  std=1.11e-03  min=-1.67e-02  max=7.84e-03


### 4.1 Variance pre-filter (conditional) + daily-gap features

Two things happen in this cell:

1. **Variance pre-filter (conditional)** — when `FEATURE_ENGINEERING_DONE = False` in the config cell, we drop near-constants and keep the top-`MAX_FEATURES_PREFILTER` columns by variance (plus all AR features). Once feature engineering / selection has been done upstream, flip the flag to `True` and this filter is skipped. Variance ≠ relevance — this is a stop-gap, not real selection.
2. **Daily-gap features (Approach 2 + 3)** — `is_post_break` is a regime dummy; `poly_movement_during_break` sums `|ΔPolymarket|` across the overnight break and captures the information accumulated while the gold feed was closed. Toggle with `HANDLE_DAILY_GAP`. The other approaches stay in Cell 10 (commented out) so you can revert.


In [10]:
# %% ── CELL 4.1 : VARIANCE PREFILTER + DAILY-GAP FEATURES ──────────────────
# ─── FIX (Operation Fixing Terrible Performance, point 3) ────────────────────
# The variance pre-filter is a STOP-GAP for the period when Polymarket features
# are still raw and high-dimensional. Once proper feature engineering and
# selection have been done upstream (flip FEATURE_ENGINEERING_DONE = True in
# the config cell), the filter is skipped and every surviving column is kept.
# NOTE: variance ≠ relevance — a slowly-moving probability that just crossed
#       a threshold can be very informative. Keeping this only until FE is in.
if FEATURE_ENGINEERING_DONE:
    print("🆗 FEATURE_ENGINEERING_DONE=True -> skipping variance prefilter.")
    X_all      = X.copy()
    # AR features are guaranteed to already be in X — no-op.
    print(f"X shape (no prefilter): {X.shape}")
else:
    vt = VarianceThreshold(threshold=1e-8)
    vt.fit(X.values)
    kept_mask  = vt.get_support()
    kept_cols  = X.columns[kept_mask]
    print(f"After VarianceThreshold: {len(kept_cols)} / {X.shape[1]} columns survive.")

    # Keep the top-N by variance (heuristic only — cheap to change later)
    variances    = X[kept_cols].var().sort_values(ascending=False)
    top_cols     = variances.head(MAX_FEATURES_PREFILTER).index.tolist()
    # Always keep AR features regardless of variance ranking
    for c in ar_cols:
        if c in X.columns and c not in top_cols:
            top_cols.append(c)

    X_all = X.copy()                 # untouched full matrix (for reference)
    X     = X[top_cols]
    print(f"X shape after prefilter : {X.shape}")

# Re-sync the polymarket column list to whatever actually survived in X.
# (Used downstream for PCA / gap features.)
polymarket_cols_kept = [c for c in X.columns if c not in ar_cols]
print(f"  polymarket cols kept: {len(polymarket_cols_kept)}  |  AR cols: {len(ar_cols)}")

# ─── FIX (Operation Fixing Terrible Performance, point 4) ────────────────────
# Bloomberg's gold feed has a ~1h daily break starting ~23:00. The first bar
# after re-open is actually a ~65-minute return carrying ALL the information
# that accumulated during the break — dropping it wastes signal. So we DO NOT
# mask it (Approach 1). Instead we combine Approach 2 + 3:
#   (2) is_post_break dummy -> lets the model learn the different regime;
#   (3) poly_movement_during_break = sum of |ΔPolymarket| during the gap
#       -> the accumulated-information signal.
# Approach 1 and Approach 4 remain in Cell 10 (commented out) so we can revert.
# This block runs BEFORE the training grid (unlike the original Cell 10, which
# was positioned after the grid and therefore had no effect on training).
if HANDLE_DAILY_GAP:
    _gap_step      = X.index.to_series().diff()
    _is_post_break = (_gap_step > pd.Timedelta(GAP_THRESHOLD)).astype(int)

    # (2) regime dummy
    X["is_post_break"] = _is_post_break.values

    # (3) polymarket movement accumulated during the gap
    # NOTE: X_raw below is the UN-FILTERED, pre-variance-prefilter Polymarket panel
    # (every Polymarket tick at original resolution). This is intentional: we want
    # to capture ALL Polymarket activity during the gap, not just the columns that
    # survived the variance filter. X_raw remains unchanged throughout Cell 4.1.
    _poly_abs_delta = X_raw.sort_index().diff().abs().sum(axis=1)
    _gap_feature    = pd.Series(0.0, index=X.index)
    _X_times = X.index.to_list()
    for _i in range(1, len(_X_times)):
        _t_prev, _t_curr = _X_times[_i-1], _X_times[_i]
        if (_t_curr - _t_prev) > pd.Timedelta(GAP_THRESHOLD):
            _mask = (_poly_abs_delta.index > _t_prev) & (_poly_abs_delta.index <= _t_curr)
            _gap_feature.iloc[_i] = float(_poly_abs_delta.loc[_mask].sum())
    X["poly_movement_during_break"] = _gap_feature.values

    # Both new features are behavioural / AR-like — keep them OUT of the PCA
    # block so Ridge sees them raw. We extend ar_cols to reflect that.
    ar_cols = list(ar_cols) + ["is_post_break", "poly_movement_during_break"]
    print(f"✅ Daily-gap handling ON.  post-break bars: {int(_is_post_break.sum())}  |  "
          f"non-zero 'poly_movement_during_break': {int((_gap_feature != 0).sum())}")
    print(f"  new X shape: {X.shape}")
else:
    print("ℹ️  Daily-gap handling OFF (HANDLE_DAILY_GAP=False).")


After VarianceThreshold: 2001 / 2340 columns survive.
X shape after prefilter : (3681, 161)
  polymarket cols kept: 150  |  AR cols: 11
✅ Daily-gap handling ON.  post-break bars: 12  |  non-zero 'poly_movement_during_break': 11
  new X shape: (3681, 163)


## 5. Build `X_traditional` (NOT used in modelling yet — stored for later)

In [11]:
# %% ── CELL 5 : X_traditional ───────────────────────────────────────────────
# ─── OPERATION ALIGN TRADITIONAL PREDICTORS ──────────────────────────────────
# Replaces the previous simplistic reindex+log-diff approach with a
# stationarity-safe, gold-clock-aligned build that:
#   (a) reindexes each Bloomberg sheet to the 5-min gold clock (master_index)
#       before differencing — so daily series show non-zero returns only at
#       their daily update bar, and 0 at all other 5-min bars;
#   (b) adds optional staleness counters (bars since last real tick);
#   (c) appends AR lags of the gold log-return (matching the polymarket block);
#   (d) aligns the final matrix to X.index (same valid rows as polymarket X).
# Reuses the `bb` dict already loaded in Cell 3 — no extra workbook I/O.
# ─────────────────────────────────────────────────────────────────────────────

# `gold.index` is the full 5-min Bloomberg gold clock — use it as master.
# `X.index`   is the valid modelling rows (polymarket × Bloomberg overlap).
X_traditional = build_X_traditional(
    bb_dict        = bb,
    target_sheet   = TARGET_SHEET,
    master_index   = gold.index,          # full 5-min Bloomberg gold clock
    align_index    = X.index,             # restrict to valid modelling rows
    max_ffill_bars = TRADITIONAL_MAX_FFILL_BARS,
    use_staleness  = TRADITIONAL_USE_STALENESS,
    ar_lags        = TRADITIONAL_AR_LAGS,
    ar_ma_windows  = AR_MA_WINDOWS,       # same rolling-mean windows as poly block
    ar_vol_windows = [12, 36],            # same rolling-std windows as poly block
    logret_bar     = gold["logret_bar"],
)

# y is the same target as the polymarket block (same gold returns, same index)
y_traditional = y.loc[X_traditional.index]

# ─── Daily-gap features for the traditional block ────────────────────────────
# Same methodology as Cell 4.1 for the polymarket block: we add two dummy
# features to X_traditional so that trad models can also learn the overnight
# break regime and the price information accumulated during the gap.
if HANDLE_DAILY_GAP:
    _trad_gap_step      = X_traditional.index.to_series().diff()
    _trad_is_post_break = (_trad_gap_step > pd.Timedelta(GAP_THRESHOLD)).astype(int)

    # (2) regime dummy — mirrors is_post_break in the poly block
    X_traditional["is_post_break"] = _trad_is_post_break.values

    # (3) traditional movement accumulated during the gap
    # NOTE: gold["logret_bar"] below is the UN-FILTERED, pre-alignment gold
    # log-return series (full Bloomberg gold clock). This is intentional: we
    # want to capture ALL gold price movement during the gap window, not just
    # the rows that survived alignment to X.index. The series is re-aligned
    # to X_traditional.index via reindex so the loop stays gap-aware.
    _gold_abs_delta = gold["logret_bar"].abs().reindex(X_traditional.index, method=None)
    _trad_gap_feature = pd.Series(0.0, index=X_traditional.index)
    _trad_times = X_traditional.index.to_list()
    for _i in range(1, len(_trad_times)):
        _t_prev, _t_curr = _trad_times[_i - 1], _trad_times[_i]
        if (_t_curr - _t_prev) > pd.Timedelta(GAP_THRESHOLD):
            _mask = (gold["logret_bar"].index > _t_prev) & (gold["logret_bar"].index <= _t_curr)
            _trad_gap_feature.iloc[_i] = float(gold["logret_bar"].loc[_mask].abs().sum())
    X_traditional["trad_movement_during_break"] = _trad_gap_feature.values

    print(f"✅ Trad daily-gap features added.  post-break bars: {int(_trad_is_post_break.sum())}  |  "
          f"non-zero 'trad_movement_during_break': {int((_trad_gap_feature != 0).sum())}")

# Column groups for downstream PCA masking
trad_ar_cols   = [c for c in X_traditional.columns if c.startswith("trad_gold_lag") or c.startswith("trad_gold_ma") or c.startswith("trad_gold_std") or c in ("is_post_break", "trad_movement_during_break")]
trad_pred_cols = [c for c in X_traditional.columns if c not in trad_ar_cols]

print(f"X_traditional shape : {X_traditional.shape}")
print(f"y_traditional shape : {y_traditional.shape}")
print(f"  predictor cols    : {len(trad_pred_cols)}  (will go through PCA if enabled)")
print(f"  AR lag cols       : {len(trad_ar_cols)}   (kept raw, bypass PCA)")
print(f"  Features preview  : {list(X_traditional.columns[:8])}")


✅ Trad daily-gap features added.  post-break bars: 12  |  non-zero 'trad_movement_during_break': 12
X_traditional shape : (3681, 25)
y_traditional shape : (3681,)
  predictor cols    : 12  (will go through PCA if enabled)
  AR lag cols       : 13   (kept raw, bypass PCA)
  Features preview  : ['crude_oil_wti_futures_price_logret', 'crude_oil_brent_futures_price_logret', 's_p_500_index_logret', 'usd_index_logret', 'vix_index_logret', 'usgg10yr_index_logret', 'crude_oil_wti_futures_price_stale', 'crude_oil_brent_futures_price_stale']


## 5.1 Dataset registry

`DATASET_SPECS` is the single place that registers every `(X, y)` pair the training loop should iterate over.  Adding a new feature set (e.g. combined polymarket + traditional) is a one-line addition here.

| Key | X | y | PCA | Non-PCA cols |
|-----|---|---|-----|--------------|
| `poly` | Polymarket + AR | gold log-return | `PCA_N_COMPONENTS` | `ar_cols` |
| `trad` | Bloomberg traditional + AR | gold log-return | `PCA_N_COMPONENTS_TRADITIONAL` | `trad_ar_cols` |

In [12]:
# %% ── CELL 5.1 : DATASET REGISTRY ──────────────────────────────────────────
# Each entry:
#   "X"               : predictor DataFrame (already stationary, aligned to y)
#   "y"               : target Series
#   "pca_n_components": number of PCA components (0 = no PCA)
#   "non_pca_cols"    : columns that skip PCA and are fed raw to the model
#                       (AR lags, gap features, staleness counters)
DATASET_SPECS = {
    "poly": {
        "X"               : X,
        "y"               : y,
        "pca_n_components": PCA_N_COMPONENTS,
        "non_pca_cols"    : ar_cols,          # global AR cols defined in Cell 4.1
    },
    "trad": {
        "X"               : X_traditional,
        "y"               : y_traditional,
        "pca_n_components": PCA_N_COMPONENTS_TRADITIONAL,
        "non_pca_cols"    : trad_ar_cols,     # trad_gold_lag* columns
    },
}

for tag, ds in DATASET_SPECS.items():
    print(f"  [{tag}]  X={ds['X'].shape}  y={ds['y'].shape}  "
          f"pca_n={ds['pca_n_components']}  non_pca={len(ds['non_pca_cols'])} cols")


  [poly]  X=(3681, 163)  y=(3681,)  pca_n=30  non_pca=13 cols
  [trad]  X=(3681, 25)  y=(3681,)  pca_n=5  non_pca=13 cols


## 6. Modelling framework (walk-forward)

Everything below is organised so that **adding / removing a model is a one-line change**:

```python
MODEL_REGISTRY = {
    "linear": make_linear,
    "rf":     make_rf,
    "lstm":   make_lstm,
}
```

A *model factory* returns an object exposing `.fit(X,y)` and `.predict(X)`. The walk-forward loop is identical for all of them; only LSTM uses the **batched retraining** path (one refit every `LSTM_TEST_BLOCK` obs).

### Cross-validation strategy
For each `(model, window_scheme)` combo we run an **expanding or fixed-size walk-forward** where at each step the model is trained on `[t-window, t-1]` (fixed) or `[start, t-1]` (expanding) and predicts `y_t`. This is the closest scikit-learn analogue to your "train on rolling window, test on training_size+1" spec. For LSTM we predict 12 steps in one go before refitting — same walk-forward, just coarser grid.

In [13]:
# %% ── CELL 6 : HELPERS — WINDOW INDICES & METRICS ──────────────────────────
def walk_forward_indices(n: int, kind: str, size: int, step: int = 1):
    '''Yield (train_idx_slice, test_idx_slice) tuples over 0..n-1.
       kind='fixed'      -> train = [i-size : i]
       kind='expanding'  -> train = [0      : i]
       test is [i : i+step].'''
    assert kind in ("fixed","expanding")
    start = size                         # first index for which we have enough history
    for i in range(start, n, step):
        if kind == "fixed":
            tr = slice(i-size, i)
        else:
            tr = slice(0, i)
        te = slice(i, min(i+step, n))
        if te.stop <= te.start:
            break
        yield tr, te

def compute_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    m = {
        "n"       : int(len(y_true)),
        "rmse"    : float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae"     : float(mean_absolute_error(y_true, y_pred)),
        "r2"      : float(r2_score(y_true, y_pred)) if len(y_true) > 1 else float("nan"),
        "dir_acc" : float(np.mean(np.sign(y_true) == np.sign(y_pred))),
        "hit_rate_nonzero" : float(np.mean((y_true != 0) & (np.sign(y_true) == np.sign(y_pred)))
                                   / max((y_true != 0).mean(), 1e-9)),
    }
    return m


In [14]:
# %% ── CELL 6.1 : MODEL FACTORIES ────────────────────────────────────────────
# ─── FIX (Operation Fixing Terrible Performance, point 2) ────────────────────
# * Ridge regularisation is now configurable via RIDGE_ALPHA.
# * Optional PCA pre-compression of the Polymarket block (NOT the AR block)
#   is added. PCA is fit on the TRAINING slice only inside each walk-forward
#   window, so there is no look-ahead leakage. Components are controlled by
#   PCA_N_COMPONENTS (set to 0/None to disable).
# * LSTM is KEPT (per user instruction). We are aware it's under-determined
#   with 161 features and 120-row windows — we track its performance as we
#   iterate on feature engineering.
# ─── UPDATE (Operation Align Traditional Predictors) ─────────────────────────
# * _pca_mask_for now accepts an explicit `non_pca_cols` list so it works for
#   both the polymarket block (ar_cols) and the traditional block (trad_ar_cols).
# * make_linear accepts `pca_n` and `non_pca_cols` kwargs so the training loop
#   can pass dataset-specific PCA settings without touching the global config.
class ScaledRegressor:
    """
    Wrapper that:
      1) standardises X (and optionally y);
      2) OPTIONALLY applies PCA to a subset of columns (Polymarket block),
         leaving the remaining columns (AR block) untouched;
      3) delegates fit/predict to a core estimator.
    Keeps the main walk-forward loop model-agnostic.
    """
    def __init__(self, core, scale_y=False,
                 pca_n_components=None, pca_col_mask=None):
        self.core          = core
        self.scale_y       = scale_y
        self.xs            = StandardScaler()
        self.ys            = StandardScaler() if scale_y else None
        # --- PCA on a subset of columns (if enabled) ---
        self.pca_n         = pca_n_components if (pca_n_components and pca_n_components > 0) else None
        self.pca_col_mask  = np.asarray(pca_col_mask) if (pca_col_mask is not None and self.pca_n) else None
        self.pca           = None

    def _transform_X(self, Xs, fit: bool):
        """Split into (PCA-block, passthrough-block), apply PCA to the former."""
        if self.pca_col_mask is None:
            return Xs
        pca_part  = Xs[:, self.pca_col_mask]
        rest_part = Xs[:, ~self.pca_col_mask]
        n_comp = min(self.pca_n, pca_part.shape[1], pca_part.shape[0])
        if n_comp < 1:
            return Xs
        if fit:
            self.pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
            pca_out  = self.pca.fit_transform(pca_part)
        else:
            pca_out  = self.pca.transform(pca_part)
        return np.hstack([pca_out, rest_part])

    def fit(self, X, y):
        Xs = self.xs.fit_transform(X)
        Xs = self._transform_X(Xs, fit=True)
        if self.ys is not None:
            ys = self.ys.fit_transform(np.asarray(y).reshape(-1,1)).ravel()
            self.core.fit(Xs, ys)
        else:
            self.core.fit(Xs, y)
        return self

    def predict(self, X):
        Xs = self.xs.transform(X)
        Xs = self._transform_X(Xs, fit=False)
        pred = self.core.predict(Xs)
        if self.ys is not None:
            pred = self.ys.inverse_transform(np.asarray(pred).reshape(-1,1)).ravel()
        return np.asarray(pred).ravel()


def _pca_mask_for(columns, non_pca_cols=None) -> np.ndarray:
    """Boolean mask: True for columns that go through PCA.

    `non_pca_cols` — columns that bypass PCA and are kept raw (AR lags, gap
    features, staleness counters).  Defaults to the global `ar_cols` so the
    polymarket path is unchanged.
    """
    _exclude = set(non_pca_cols) if non_pca_cols is not None else set(ar_cols)
    return np.asarray([c not in _exclude for c in columns])


def make_linear(feature_columns=None, pca_n=None, non_pca_cols=None):
    """Ridge + optional PCA.

    `pca_n`        — override PCA_N_COMPONENTS for this call (used by the
                     dataset registry loop to pass PCA_N_COMPONENTS_TRADITIONAL
                     for the traditional block).
    `non_pca_cols` — columns that bypass PCA; defaults to global `ar_cols`.
    """
    mask   = _pca_mask_for(feature_columns, non_pca_cols=non_pca_cols) if feature_columns is not None else None
    _pca_n = pca_n if pca_n is not None else PCA_N_COMPONENTS
    return ScaledRegressor(
        Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_STATE),
        pca_n_components = _pca_n,
        pca_col_mask     = mask,
    )

def make_rf(feature_columns=None, pca_n=None, non_pca_cols=None):
    # Trees are scale-invariant — no wrapper, no PCA needed.
    # Extra kwargs accepted to keep factory signatures uniform.
    return RandomForestRegressor(
        n_estimators=RF_N_ESTIMATORS, max_depth=None,
        min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE,
    )

def _build_lstm(n_features: int, seq_len: int = LSTM_SEQ_LEN):
    model = Sequential([
        Input(shape=(seq_len, n_features)),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1, activation="linear"),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

class LSTMRegressor:
    """LSTM with internal X/y scaling and a sliding-window transformer.
       predict(X) expects a 2-D array where row i corresponds to the "current" timestep;
       the model looks at the previous seq_len rows internally.

       Note: kept intentionally WITHOUT PCA — the LSTM already benefits from
       the raw temporal structure, and compressing features before the sequence
       window would destroy that. We track its performance as feature
       engineering progresses.
    """
    def __init__(self, seq_len=LSTM_SEQ_LEN, epochs=LSTM_EPOCHS, batch=LSTM_BATCH):
        self.seq_len, self.epochs, self.batch = seq_len, epochs, batch
        self.xs, self.ys = StandardScaler(), StandardScaler()
        self.model = None
        self._last_train_X = None   # used to build sequences that span the train→test boundary
    def _seq(self, Xs, ys=None):
        X_out, y_out = [], []
        for i in range(self.seq_len, len(Xs)):
            X_out.append(Xs[i-self.seq_len:i])
            if ys is not None:
                y_out.append(ys[i])
        X_out = np.asarray(X_out)
        return (X_out, np.asarray(y_out)) if ys is not None else X_out
    def fit(self, X, y):
        X = np.asarray(X); y = np.asarray(y).reshape(-1,1)
        Xs = self.xs.fit_transform(X)
        ys = self.ys.fit_transform(y).ravel()
        Xseq, yseq = self._seq(Xs, ys)
        self.model = _build_lstm(X.shape[1], self.seq_len)
        es = EarlyStopping(patience=3, restore_best_weights=True, monitor="loss")
        self.model.fit(Xseq, yseq, epochs=self.epochs, batch_size=self.batch,
                       verbose=0, callbacks=[es])
        self._last_train_X = Xs[-self.seq_len:]
        return self
    def predict(self, X):
        X = np.asarray(X)
        Xs = self.xs.transform(X)
        Xs_ext = np.vstack([self._last_train_X, Xs]) if self._last_train_X is not None else Xs
        preds = []
        for i in range(self.seq_len, len(Xs_ext)):
            window = Xs_ext[i-self.seq_len:i][None, ...]
            preds.append(self.model.predict(window, verbose=0)[0,0])
        preds = np.asarray(preds)
        return self.ys.inverse_transform(preds.reshape(-1,1)).ravel()

def make_lstm(feature_columns=None, pca_n=None, non_pca_cols=None):
    # Extra kwargs accepted to keep factory signatures uniform.
    if not _TF_AVAILABLE:
        return None
    return LSTMRegressor()

MODEL_REGISTRY = {
    "linear": make_linear,
    "rf":     make_rf,
    "lstm":   make_lstm,
}


### 6.2 Walk-forward runner

For `linear` and `rf` we use `step=1` (one-step-ahead, the finest grid possible). For `lstm` we use `step=LSTM_TEST_BLOCK` (=12, ~1 hour per refit) per your spec 6.1. The same helper covers both.

In [15]:
# %% ── CELL 6.2 : WALK-FORWARD RUNNER ────────────────────────────────────────
# ─── UPDATE (Operation Align Traditional Predictors) ─────────────────────────
# Added `pca_n_components` and `non_pca_cols` kwargs so the outer dataset loop
# can pass dataset-specific PCA settings.  The polymarket path is unchanged
# (those kwargs default to None, which makes each factory fall back to globals).
def walk_forward(model_name: str, scheme_name: str, X: pd.DataFrame, y: pd.Series,
                 pca_n_components: int = None, non_pca_cols=None):
    kind, size = WINDOW_SCHEMES[scheme_name]
    step = LSTM_TEST_BLOCK if model_name == "lstm" else 1
    y_true, y_pred, ts_pred = [], [], []
    train_time_s = 0.0
    n = len(X)
    iters = list(walk_forward_indices(n, kind, size, step=step))
    feature_columns = list(X.columns)
    for k, (tr, te) in enumerate(iters):
        factory = MODEL_REGISTRY[model_name]
        try:
            mdl = factory(feature_columns=feature_columns,
                          pca_n=pca_n_components,
                          non_pca_cols=non_pca_cols)
        except TypeError:
            mdl = factory()
        if mdl is None:
            return None
        _t0 = time.perf_counter()
        mdl.fit(X.iloc[tr].values, y.iloc[tr].values)
        train_time_s += time.perf_counter() - _t0
        p  = mdl.predict(X.iloc[te].values)
        y_true.extend(y.iloc[te].values.tolist())
        y_pred.extend(np.asarray(p).tolist())
        ts_pred.extend(y.iloc[te].index.tolist())
        if (k % 50 == 0):
            print(f"  [{model_name}/{scheme_name}] step {k+1}/{len(iters)} (train={tr.stop-tr.start}, test={te.stop-te.start})")
    return {
        "y_true": np.asarray(y_true),
        "y_pred": np.asarray(y_pred),
        "timestamps": ts_pred,
        "final_model": mdl,
        "train_time_s": train_time_s,
    }


## 7. Logging and model persistence

- `Results/runs_log.txt` — one human-readable row per `(model, window, data-date)` run.
- `Results/runs_log.jsonl` — the same info as JSON for reproducibility / downstream comparison across weeks.
- `Models/<MODEL>_<WINDOW>_<DATA-DATE>.(pkl|keras)` — the **final-step** fitted model (trained on the last available window), so you can reload it without rerunning the whole walk-forward.

**"Retrain only if data changed"** logic:
- We compute `sha1` of the input gold panel CSV (content, not filename) and store it inside the sidecar JSON. On the next run, if a cached model for that `(model, window, data-hash)` triple exists and `FORCE_RETRAIN is False`, we just reload.

In [16]:
# %% ── CELL 7 : PERSISTENCE HELPERS ─────────────────────────────────────────
def file_sha1(path: Path, bufsize=1<<20) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(bufsize), b""):
            h.update(chunk)
    return h.hexdigest()

DATA_HASH = file_sha1(panel_path)
print("Data SHA1:", DATA_HASH[:12], "…")

# ─── UPDATE (Operation Align Traditional Predictors) ─────────────────────────
# `artefact_paths` now accepts `dataset_tag` ("poly" | "trad") and `pca_n` so
# that polymarket and traditional artefacts never share file names.
# Pattern: <MODEL>_<WINDOW>_<DATASET>_h<horizonMin>m_a<ridge>_p<pca>_f<nFeat>_<dataDate>
def _fmt_alpha(a: float) -> str:
    return f"{a:g}".replace(".", "p")        # e.g. 10.0 -> '10', 0.5 -> '0p5'

def artefact_paths(model_name: str, scheme: str, data_date: str, n_features: int,
                   dataset_tag: str = "poly", pca_n: int = None) -> tuple[Path, Path]:
    """Return (model_path, metadata_path) for a given run configuration.

    `dataset_tag` is embedded in the stem so polymarket ("poly") and
    traditional ("trad") artefacts never collide even when all other
    parameters are identical.
    `pca_n` overrides PCA_N_COMPONENTS in the stem (used for the traditional
    block which uses PCA_N_COMPONENTS_TRADITIONAL).
    """
    _pca = pca_n if pca_n is not None else PCA_N_COMPONENTS
    tokens = [
        model_name.upper(),
        scheme,
        dataset_tag,
        f"h{RETURN_HORIZON_MIN}m",
        f"a{_fmt_alpha(RIDGE_ALPHA)}",
        f"p{_pca or 0}",
        f"f{n_features}",
        data_date,
    ]
    stem = "_".join(tokens)
    ext = ".keras" if model_name == "lstm" else ".pkl"
    return MODELS_DIR / f"{stem}{ext}", RESULTS_DIR / f"{stem}.json"

def cached_run_valid(meta_path: Path, data_hash: str) -> bool:
    if not meta_path.exists():
        return False
    try:
        meta = json.loads(meta_path.read_text())
        return meta.get("data_sha1") == data_hash and not FORCE_RETRAIN
    except Exception:
        return False

def save_artefacts(model_name, scheme, data_date, data_hash, metrics, result,
                   X_cols, train_time_s=None, dataset_tag="poly", pca_n=None):
    """Persist model + metadata and append to the master run logs.

    `dataset_tag` and `pca_n` are forwarded to `artefact_paths` and recorded
    in the sidecar JSON so runs are fully reproducible from the artefact alone.
    """
    n_features = len(X_cols)
    _pca_n     = pca_n if pca_n is not None else PCA_N_COMPONENTS
    mdl_path, meta_path = artefact_paths(model_name, scheme, data_date, n_features,
                                          dataset_tag=dataset_tag, pca_n=_pca_n)
    mdl = result["final_model"]
    # --- save final-step fitted model ---
    if model_name == "lstm":
        mdl.model.save(mdl_path)
        joblib.dump({"xs": mdl.xs, "ys": mdl.ys, "seq_len": mdl.seq_len,
                     "last_train_X": mdl._last_train_X},
                    mdl_path.with_suffix(".scalers.pkl"))
    else:
        joblib.dump(mdl, mdl_path)
    # --- side-car metadata (expanded) ---
    meta = {
        "model"            : model_name,
        "window"           : scheme,
        "dataset_tag"      : dataset_tag,
        "data_date"        : data_date,
        "data_sha1"        : data_hash,
        "run_utc"          : dt.datetime.utcnow().isoformat(timespec="seconds"),
        "train_time_s"     : round(train_time_s, 3) if train_time_s is not None else None,
        "n_features"       : n_features,
        "feature_sample"   : X_cols[:15],
        "metrics"          : metrics,
        "config"           : {
            # ─── target / horizon ────────────────────────────────────────────
            "bar_minutes"              : BAR_MINUTES,
            "return_horizon_min"       : RETURN_HORIZON_MIN,
            "horizon_steps"            : HORIZON_STEPS,
            # ─── AR block ────────────────────────────────────────────────────
            "ar_lags"                  : AR_LAGS,
            "ar_ma_windows"            : AR_MA_WINDOWS,
            # ─── regularisation / dim-reduction ──────────────────────────────
            "ridge_alpha"              : RIDGE_ALPHA,
            # pca_applied is False when PCA is disabled (pca_n = 0/None);
            # pca_n is only stored when PCA is actually applied.
            "pca_applied"              : bool(_pca_n and _pca_n > 0),
            **({"pca_n": _pca_n} if (_pca_n and _pca_n > 0) else {}),
            # ─── feature-engineering gate / prefilter ────────────────────────
            "feature_engineering_done" : FEATURE_ENGINEERING_DONE,
            "prefilter_topN"           : MAX_FEATURES_PREFILTER if not FEATURE_ENGINEERING_DONE else None,
            # ─── LSTM / RF specifics ─────────────────────────────────────────
            "lstm_seq_len"             : LSTM_SEQ_LEN,
            "lstm_block"               : LSTM_TEST_BLOCK,
            "lstm_epochs"              : LSTM_EPOCHS,
            "lstm_batch"               : LSTM_BATCH,
            "rf_estimators"            : RF_N_ESTIMATORS,
            "random_state"             : RANDOM_STATE,
            # ─── gap handling ────────────────────────────────────────────────
            "handle_daily_gap"         : HANDLE_DAILY_GAP,
            "gap_threshold"            : GAP_THRESHOLD,
            # ─── traditional-specific ────────────────────────────────────────
            "dataset_tag"              : dataset_tag,
            "trad_max_ffill_bars"      : TRADITIONAL_MAX_FFILL_BARS if dataset_tag == "trad" else None,
            "trad_use_staleness"       : TRADITIONAL_USE_STALENESS  if dataset_tag == "trad" else None,
            "trad_ar_lags"             : TRADITIONAL_AR_LAGS        if dataset_tag == "trad" else None,
        },
    }
    meta_path.write_text(json.dumps(meta, indent=2, default=str))
    # --- append to master log ---
    train_time_str = f"{train_time_s:.1f}s" if train_time_s is not None else "N/A"
    with open(RESULTS_DIR / "runs_log.txt", "a") as f:
        f.write(
            f"{meta['run_utc']}  {model_name:7s}  {scheme:10s}  {dataset_tag:5s}  "
            f"h={RETURN_HORIZON_MIN}m  alpha={RIDGE_ALPHA:g}  pca={_pca_n or 0}  "
            f"f={n_features}  data={data_date}  rmse={metrics['rmse']:.5e}  "
            f"mae={metrics['mae']:.5e}  r2={metrics['r2']:+.4f}  "
            f"dir_acc={metrics['dir_acc']:.3f}  train_time={train_time_str}\n")
    with open(RESULTS_DIR / "runs_log.jsonl", "a") as f:
        f.write(json.dumps(meta, default=str) + "\n")
    return meta_path, mdl_path


Data SHA1: 98716effaf1e …


## 8. Run the full grid

`MODELS × WINDOW_SCHEMES`. Cached artefacts are reused if the data hasn't changed.

In [17]:
# %% ── CELL 8 : RUN THE GRID ─────────────────────────────────────────────────
# ─── UPDATE (Operation Align Traditional Predictors) ─────────────────────────
# Outer loop now iterates over DATASET_SPECS so the same model grid runs on
# both the polymarket block ("poly") and the traditional Bloomberg block ("trad").
# Artefact names include the dataset tag to prevent file-name collisions.
MODELS_TO_RUN = ["linear", "rf", "lstm"]   # edit freely; registry is extensible
all_results = []

for dataset_tag, ds in DATASET_SPECS.items():
    X_ds         = ds["X"]
    y_ds         = ds["y"]
    _pca_n       = ds["pca_n_components"]
    _non_pca     = ds.get("non_pca_cols", ar_cols)
    _N_FEATURES  = X_ds.shape[1]

    print(f"\n{'='*60}")
    print(f"  Dataset: {dataset_tag}  |  X={X_ds.shape}  |  pca_n={_pca_n}")
    print(f"{'='*60}")

    for model_name in MODELS_TO_RUN:
        if model_name == "lstm" and not _TF_AVAILABLE:
            print(f"⏭  Skipping LSTM (TensorFlow not available).")
            continue
        for scheme in WINDOW_SCHEMES:
            mdl_path, meta_path = artefact_paths(
                model_name, scheme, panel_date, _N_FEATURES,
                dataset_tag=dataset_tag, pca_n=_pca_n,
            )
            if cached_run_valid(meta_path, DATA_HASH):
                meta = json.loads(meta_path.read_text())
                train_time_str = (f"{meta['train_time_s']:.1f}s"
                                  if meta.get("train_time_s") is not None else "N/A")
                print(f"✓ Cached: {dataset_tag}/{model_name}/{scheme} — "
                      f"rmse={meta['metrics']['rmse']:.3e}  "
                      f"dir_acc={meta['metrics']['dir_acc']:.3f}  "
                      f"train_time={train_time_str}")
                all_results.append(meta)
                continue

            print(f"▶ Training {dataset_tag}/{model_name}/{scheme} …  "
                  f"(horizon={RETURN_HORIZON_MIN}m  alpha={RIDGE_ALPHA:g}  "
                  f"pca={_pca_n or 0}  features={_N_FEATURES})")
            res = walk_forward(model_name, scheme, X_ds, y_ds,
                               pca_n_components=_pca_n, non_pca_cols=_non_pca)
            if res is None:
                continue
            metrics      = compute_metrics(res["y_true"], res["y_pred"])
            train_time_s = res["train_time_s"]
            save_artefacts(
                model_name, scheme, panel_date, DATA_HASH, metrics, res,
                list(X_ds.columns), train_time_s,
                dataset_tag=dataset_tag, pca_n=_pca_n,
            )
            all_results.append({
                "model"       : model_name,
                "window"      : scheme,
                "dataset_tag" : dataset_tag,
                "data_date"   : panel_date,
                "n_features"  : _N_FEATURES,
                "metrics"     : metrics,
                "train_time_s": train_time_s,
            })
            print(f"   ✓ rmse={metrics['rmse']:.3e}  mae={metrics['mae']:.3e}  "
                  f"r2={metrics['r2']:+.4f}  dir_acc={metrics['dir_acc']:.3f}  "
                  f"train_time={train_time_s:.1f}s")



  Dataset: poly  |  X=(3681, 163)  |  pca_n=30
▶ Training poly/linear/fixed120 …  (horizon=5m  alpha=10  pca=30  features=163)
  [linear/fixed120] step 1/3561 (train=120, test=1)
  [linear/fixed120] step 51/3561 (train=120, test=1)
  [linear/fixed120] step 101/3561 (train=120, test=1)
  [linear/fixed120] step 151/3561 (train=120, test=1)
  [linear/fixed120] step 201/3561 (train=120, test=1)
  [linear/fixed120] step 251/3561 (train=120, test=1)
  [linear/fixed120] step 301/3561 (train=120, test=1)
  [linear/fixed120] step 351/3561 (train=120, test=1)
  [linear/fixed120] step 401/3561 (train=120, test=1)
  [linear/fixed120] step 451/3561 (train=120, test=1)
  [linear/fixed120] step 501/3561 (train=120, test=1)
  [linear/fixed120] step 551/3561 (train=120, test=1)
  [linear/fixed120] step 601/3561 (train=120, test=1)
  [linear/fixed120] step 651/3561 (train=120, test=1)
  [linear/fixed120] step 701/3561 (train=120, test=1)
  [linear/fixed120] step 751/3561 (train=120, test=1)
  [linear/f

## 9. Summary table

In [2]:
# %% ── CELL 9 : SUMMARY ─────────────────────────────────────────────────────
import importlib, sys
sys.path.insert(0, "./Functions")
import save_summary_table as _sst
importlib.reload(_sst)
from save_summary_table import build_summary_table

summary = build_summary_table(
    all_results, RETURN_HORIZON_MIN, RIDGE_ALPHA, PCA_N_COMPONENTS,
    panel_date, RESULTS_DIR
)


NameError: name 'all_results' is not defined

## 10. Alternative daily-gap approaches (commented out)

The **active** gap handling (Approach 2 + 3) lives in Cell 4.1 so it is applied before training. This section keeps Approach 1 (masking) and Approach 4 (per-session standardisation) commented out for easy reverting / experimentation.


In [19]:
# %% ── CELL 10 : BREAK-HANDLING (alternative approaches kept for reference) ──
# The ACTIVE gap-handling code (Approach 2 + 3, user-selected) now lives in
# Cell 4.1 so that it is applied BEFORE training. This cell keeps the other
# two approaches commented out so you can revert / experiment later without
# rewriting anything.
# Active approach (2 + 3) is applied in Cell 4.1 when HANDLE_DAILY_GAP=True.
# The alternatives below are kept commented out for easy toggling.

# # -------- Approach 1: drop the first bar after each break > 1 hour --------
# gap_mask = X.index.to_series().diff() > pd.Timedelta(GAP_THRESHOLD)
# X = X.loc[~gap_mask]
# y = y.loc[X.index]

# # -------- Approach 4: per-session standardisation of y --------------------
# sessions = (X.index.to_series().diff() > pd.Timedelta(GAP_THRESHOLD)).cumsum()
# y = y.groupby(sessions).transform(lambda s: (s - s.mean()) / (s.std() + 1e-12))
print("Cell 10: see Cell 4.1 for the active gap-handling logic (Approach 2 + 3).")


Cell 10: see Cell 4.1 for the active gap-handling logic (Approach 2 + 3).


## 11. How to reload a model later (no retraining)

In [20]:
# %% ── CELL 11 : MODEL RELOAD EXAMPLE ───────────────────────────────────────
# Example — load the expanding-window Linear model trained on today's data:
# mdl_path, meta_path = artefact_paths("linear", "expanding", panel_date)
# model = joblib.load(mdl_path)
# meta  = json.loads(meta_path.read_text())
# print("Reloaded:", meta["model"], meta["window"], "metrics:", meta["metrics"])
# # For LSTM:
# # from tensorflow.keras.models import load_model
# # keras_model = load_model(mdl_path)
# # scalers = joblib.load(mdl_path.with_suffix(".scalers.pkl"))
